# XPDE reproducible Colab training

CPU-only CatBoost training for `GOLDm#` M5. This notebook verifies the exact Git commit, dataset hash, tests, candidate gate, and artifact checksums before copying results to Drive.

In [ ]:
REPOSITORY = "https://github.com/badaruddinl/xpde.git"
GIT_COMMIT = "REPLACE_WITH_EXACT_COMMIT"
DRIVE_DATASET = "/content/drive/MyDrive/xpde/datasets/validated/goldm_m5.csv"
DRIVE_MANIFEST = "/content/drive/MyDrive/xpde/datasets/validated/goldm_m5.manifest.json"
DRIVE_ARTIFACTS = "/content/drive/MyDrive/xpde/artifacts/candidates"
ITERATIONS = 250
FOLDS = 4

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, pathlib, shutil, subprocess

repo = pathlib.Path("/content/xpde")
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", REPOSITORY, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", GIT_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == GIT_COMMIT, (actual_commit, GIT_COMMIT)

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(repo / "ml/requirements-training.lock.txt")], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "--no-deps", "-e", str(repo / "ml")], check=True)
subprocess.run(["python", "-m", "pytest", "-q", str(repo / "ml/tests")], check=True)
subprocess.run(["cargo", "fmt", "--all", "--", "--check"], cwd=repo, check=True)
subprocess.run(["cargo", "test", "--workspace"], cwd=repo, check=True)
subprocess.run(["npm", "ci"], cwd=repo, check=True)
subprocess.run(["npm", "test"], cwd=repo, check=True)

In [ ]:
import hashlib, json

local_dataset = pathlib.Path("/content/goldm_m5.csv")
local_manifest = pathlib.Path("/content/goldm_m5.manifest.json")
shutil.copy2(DRIVE_DATASET, local_dataset)
shutil.copy2(DRIVE_MANIFEST, local_manifest)
dataset_manifest = json.loads(local_manifest.read_text())
actual_sha256 = hashlib.sha256(local_dataset.read_bytes()).hexdigest()
assert actual_sha256 == dataset_manifest["sha256"], "dataset SHA-256 mismatch"
assert dataset_manifest["symbol"] == "GOLDm#"
assert dataset_manifest["timeframe"] == "M5"
assert dataset_manifest["contains_incomplete_bar"] is False
dataset_manifest

In [ ]:
run_root = pathlib.Path("/content/xpde-artifacts")
if run_root.exists():
    shutil.rmtree(run_root)
run_root.mkdir(parents=True)
command = [
    "python", "-m", "xpde_ml.train_catboost", str(local_dataset),
    "--output", str(run_root),
    "--iterations", str(ITERATIONS),
    "--folds", str(FOLDS),
    "--no-register",
]
training_env = {**os.environ, "XPDE_TRAINING_RUNTIME": "google-colab-cpu"}
subprocess.run(command, cwd=repo / "ml", env=training_env, check=True)

In [ ]:
import sys
sys.path.insert(0, str(repo / "ml"))
from xpde_ml.model_inference import verify_artifact_checksums

verify_artifact_checksums(run_root)
candidate_manifest = json.loads((run_root / "manifest.json").read_text())
assert candidate_manifest["schema_version"] >= 3
assert candidate_manifest["training_environment"]["training_git_commit"] == GIT_COMMIT
assert candidate_manifest["training_environment"]["training_git_dirty"] is False
assert candidate_manifest["training_environment"]["training_runtime"] == "google-colab-cpu"
assert candidate_manifest["source_dataset"]["sha256"] == actual_sha256
candidate_manifest["eligible_for_shadow"]

In [ ]:
destination = pathlib.Path(DRIVE_ARTIFACTS) / candidate_manifest["model_id"]
if destination.exists():
    raise FileExistsError(f"immutable candidate already exists: {destination}")
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_root, destination)
archive = shutil.make_archive(str(destination), "zip", root_dir=run_root)
print({"candidate": str(destination), "archive": archive, "eligible_for_shadow": candidate_manifest["eligible_for_shadow"]})